# fea-surrogate — training on the swept dataset

Trains on the dataset generated by `fea_surrogate.generate.sweep`: 2960 samples
across six component families, solved with linear elastic FEA, with **load
location in the features** and a **dimensionless target**.

**Runtime → Change runtime type → T4 GPU.** Cells run 1 → 10 in order.

### The question this run answers

SFEM gave a conformal band of **×/÷ 19.75**. That dataset encodes load
*magnitude* but never where the load lands, so samples differing only in load
placement have identical inputs and different answers, and no model can beat
their mean. If that was the binding constraint, this band should come in
materially tighter.

If it does not, the problem is the model or the features rather than the data —
worth knowing before generating twenty thousand more samples.

Two things change at once here, so the comparison is indicative rather than
controlled: the dataset, and a graph-level head predicting the peak directly
instead of taking a maximum over per-node predictions. The second is the better
model for the scalar `SurrogateScreening/v1` actually carries.


## 1 · Setup

In [ ]:
import os, sys, subprocess, json, hashlib, datetime, math
try:
    import torch_geometric  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch-geometric"], check=True)
import torch, numpy as np, torch.nn as nn
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4")

## 2 · Data

Finds `sweep.tar` or an uploaded `sweep/` folder wherever it is — the Colab
filesystem or Drive. A file uploaded straight into the notebook lives on the
runtime and is lost when it disconnects; Drive survives that.

In [ ]:
import glob, shutil

DEST = '/content/sweep'
if not os.path.exists(f'{DEST}/manifest.json'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('no Drive:', exc)

    tars = (glob.glob('/content/*.tar')
            + glob.glob('/content/**/sweep.tar', recursive=True)
            + glob.glob('/content/drive/MyDrive/**/sweep.tar', recursive=True))
    folders = [os.path.dirname(p) for p in
               glob.glob('/content/drive/MyDrive/**/manifest.json', recursive=True)]

    if tars:
        print('extracting', tars[0])
        subprocess.run(['tar', '-xf', tars[0], '-C', '/content/'], check=True)
    elif folders:
        print('copying', folders[0])
        shutil.copytree(folders[0], DEST, dirs_exist_ok=True)
    else:
        raise SystemExit('no sweep.tar or sweep folder found. /content holds: '
                         + ', '.join(sorted(os.listdir('/content'))[:30]))

files = sorted(os.listdir(DEST))
print(f'{len(files)} files:', files[:4], '...')
print(json.load(open(f'{DEST}/manifest.json'))['samples'], 'samples')

## 3 · Reader

The repository's `fea_surrogate.generate.dataset`, inlined. Pure numpy, so
nothing has to be cloned into Colab.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

@dataclass(frozen=True)
class SweptSample:
    family: str
    x: np.ndarray
    y_nodal: np.ndarray
    tetrahedra: np.ndarray
    tau: float
    peak: float
    poisson: float
    force_n: float
    length_mm: float

    def edge_index(self) -> np.ndarray:
        pairs = np.array([(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)], dtype=np.int64)
        edges = np.unique(np.sort(self.tetrahedra[:, pairs].reshape(-1, 2), axis=1), axis=0)
        return np.concatenate([edges, edges[:, ::-1]]).T

class SweptDataset:
    def __init__(self, root):
        self.root = Path(root)
        self.manifest = json.loads((self.root / "manifest.json").read_text())
        self.shards = [self.root / n for n in self.manifest["shards"]]

    def __len__(self): return int(self.manifest["samples"])

    @property
    def set_sha256(self): return str(self.manifest["set_sha256"])

    def __iter__(self) -> Iterator[SweptSample]:
        for shard in self.shards:
            with np.load(shard, allow_pickle=False) as d:
                no, to = d["node_offsets"], d["tet_offsets"]
                for i in range(len(d["tau"])):
                    lo, hi = no[i], no[i+1]
                    yield SweptSample(
                        family=str(d["family"][i]), x=d["x"][lo:hi],
                        y_nodal=d["y_nodal"][lo:hi],
                        tetrahedra=d["tetrahedra"][to[i]:to[i+1]] - lo,
                        tau=float(d["tau"][i]), peak=float(d["peak"][i]),
                        poisson=float(d["poisson"][i]),
                        force_n=float(d["force_n"][i]),
                        length_mm=float(d["length_mm"][i]))

data = SweptDataset('/content/sweep')
print(len(data), 'samples |', data.set_sha256[:16], '...')

## 4 · Load into memory, split by family

Stratified, so every family appears in calibration. A band measured on a
calibration set missing a family says nothing about that family.

Nothing is dropped. An earlier run capped meshes at 6000 nodes and silently
removed 59% of the flange family, because flange meshes are the densest — a
selection effect that thinned one family's calibration set to 41 samples and
biased it toward its simpler geometries. Memory is bounded by batch size
instead, which costs time rather than data.

In [ ]:
from torch_geometric.data import Data
from collections import defaultdict

graphs = []
for s in data:
    g = Data(x=torch.tensor(s.x, dtype=torch.float),
             edge_index=torch.tensor(s.edge_index(), dtype=torch.long),
             y=torch.tensor([math.log(s.tau)], dtype=torch.float))
    g.family = s.family
    g.tau = s.tau
    graphs.append(g)

sizes = np.array([g.num_nodes for g in graphs])
print(f"loaded {len(graphs)} graphs, none dropped; "
      f"nodes median {int(np.median(sizes))} max {sizes.max()}")

by_family = defaultdict(list)
for i, g in enumerate(graphs):
    by_family[g.family].append(i)

rng = np.random.default_rng(0)
train_idx, cal_idx = [], []
for fam, idx in by_family.items():
    idx = np.array(idx); rng.shuffle(idx)
    cut = int(0.8 * len(idx))
    train_idx += idx[:cut].tolist(); cal_idx += idx[cut:].tolist()

train_ds = [graphs[i] for i in train_idx]
val_ds = [graphs[i] for i in cal_idx]
print(f"train {len(train_ds)}  calibration {len(val_ds)}")
print({f: len([i for i in cal_idx if graphs[i].family == f]) for f in by_family})

## 5 · Normaliser

Plain `log`, not `log1p`. A peak is a maximum and is never zero, and `log1p(x)`
approximates `x` for small values, which turns the transform into the identity
exactly where compression is needed. That mistake cost a training run on the
SFEM data.

In [ ]:
ys = torch.cat([g.y for g in train_ds])
G_MEAN, G_STD = ys.mean().item(), ys.std().item()
encode = lambda t: (t - G_MEAN) / (G_STD + 1e-5)
decode = lambda t: t * (G_STD + 1e-5) + G_MEAN     # returns log tau
print(f"log tau: mean {G_MEAN:.3f} std {G_STD:.3f}")
print(f"tau spans {math.exp(ys.min()):.2f} .. {math.exp(ys.max()):.0f}")

## 6 · Baseline to beat

Geometric summaries only — no message passing, no notion of where the load is.
If the GNN cannot clear this, the GNN is the problem.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

def summarise(g):
    pos = g.x[:, :3]
    ext = (pos.max(0).values - pos.min(0).values)
    loaded = g.x[:, 8] > 0.5
    fixed = g.x[:, 7] > 0.5
    centre = pos.mean(0)
    lc = pos[loaded].mean(0) if loaded.any() else centre
    fc = pos[fixed].mean(0) if fixed.any() else centre
    return torch.cat([
        ext, ext.prod().reshape(1), torch.tensor([float(len(pos))]),
        (lc - fc).abs(),                       # lever arm: load to support
        torch.tensor([float(loaded.sum()), float(fixed.sum())]),
        g.x[0, 12].reshape(1),                 # poisson
    ]).numpy()

Xtr = np.stack([summarise(g) for g in train_ds]); ytr = np.array([g.y.item() for g in train_ds])
Xva = np.stack([summarise(g) for g in val_ds]);   yva = np.array([g.y.item() for g in val_ds])
base = GradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
base_mae = mean_absolute_error(yva, base.predict(Xva))
print(f"BASELINE log-space MAE {base_mae:.4f}  (band equivalent x/÷ {math.exp(2*base_mae):.1f})")

## 7 · Save location

In [ ]:
try:
    OUT = '/content/drive/MyDrive/fea-surrogate'
    os.makedirs(OUT, exist_ok=True)
except Exception:
    OUT = '/content'
CKPT = f'{OUT}/sweep_gnn.pt'
print("saving to", CKPT)

## 8 · Model

A graph-level head. The peak is predicted directly rather than taken as a
maximum over per-node predictions, because the peak is the scalar
`SurrogateScreening/v1` carries and a max over noisy node estimates is a worse
estimator of it than a pooled read of the whole graph.

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool

dev = "cuda" if torch.cuda.is_available() else "cpu"
IN_DIM = train_ds[0].x.shape[1]

class PeakGNN(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=5):
        super().__init__()
        self.inp = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList([SAGEConv(hidden, hidden) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.SiLU(),
                                  nn.Linear(hidden, 1))
    def forward(self, d):
        h = self.inp(d.x)
        for c, n in zip(self.convs, self.norms):
            h = h + torch.nn.functional.silu(n(c(h, d.edge_index)))
        pooled = torch.cat([global_mean_pool(h, d.batch),
                            global_max_pool(h, d.batch)], dim=1)
        return self.head(pooled).reshape(-1)

model = PeakGNN(IN_DIM).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
tl = DataLoader(train_ds, batch_size=8, shuffle=True)
vl = DataLoader(val_ds, batch_size=8)
print(f"in_dim {IN_DIM}  device {dev}  params {sum(p.numel() for p in model.parameters()):,}")

## 9 · Train

In [ ]:
EPOCHS = 40
best = float('inf')
for ep in range(EPOCHS):
    model.train(); tot = seen = 0
    for b in tl:
        b = b.to(dev); opt.zero_grad()
        loss = torch.nn.functional.smooth_l1_loss(model(b), encode(b.y))
        loss.backward(); opt.step()
        tot += loss.item() * b.num_graphs; seen += b.num_graphs
    model.eval(); verr = []
    with torch.no_grad():
        for b in vl:
            b = b.to(dev)
            verr.append((decode(model(b)) - b.y).abs().cpu())
    vmae = torch.cat(verr).mean().item()
    if vmae < best:
        best = vmae; torch.save(model.state_dict(), CKPT)
    if ep % 4 == 0 or ep == EPOCHS - 1:
        print(f"epoch {ep+1:3d}/{EPOCHS}  train {tot/seen:.4f}  val log-MAE {vmae:.4f}"
              f"  (x/÷ {math.exp(2*vmae):.2f}){'  *' if vmae == best else ''}")
print(f"\nbest val log-MAE {best:.4f}   baseline was {base_mae:.4f}")

## 10 · Calibrate per family, and export

One shared band is not honest here. The previous run measured 0.90 coverage
overall while covering welded T joints 0.58 of the time and pulleys 1.00: the
average was reached by over-covering five families and badly under-covering
one. Each family now gets the band its own residuals earned, which is what
gives a per-family coverage guarantee rather than an aggregate one.

In [ ]:
model.load_state_dict(torch.load(CKPT))
model.eval()
preds, truths, fams = [], [], []
with torch.no_grad():
    for b in vl:
        bb = b.to(dev)
        preds.append(decode(model(bb)).cpu()); truths.append(b.y.cpu())
        fams.extend(b.family if isinstance(b.family, list) else [b.family])
preds = torch.cat(preds).numpy(); truths = torch.cat(truths).numpy()
fams = np.array(fams)
res = np.abs(preds - truths)          # already log space
ALPHA = 0.10

def conformal(values):
    m = len(values)
    if m < 20:
        return None                    # too few to make a coverage claim
    q = min(1.0, np.ceil((m + 1) * (1 - ALPHA)) / m)
    return float(np.quantile(values, q, method="higher"))

shared = conformal(res)
print(f"shared band  x/÷ {math.exp(shared):.2f}  coverage {float((res <= shared).mean()):.3f}")
print(f"SFEM comparison: x/÷ 19.75\n")

bands = {}
print("per family:")
for f in sorted(set(fams)):
    m = fams == f
    q = conformal(res[m])
    if q is None:
        print(f"  {f:14s} n={int(m.sum()):3d}  too few to calibrate; uses widest band")
        continue
    bands[f] = {"log_half_width": q, "band_factor": math.exp(q), "set_size": int(m.sum())}
    print(f"  {f:14s} n={int(m.sum()):3d}  own band x/÷ {math.exp(q):5.2f}"
          f"   coverage {float((res[m] <= q).mean()):.2f}"
          f"   (shared band gave {float((res[m] <= shared).mean()):.2f})")

widest = max(b["band_factor"] for b in bands.values())
print(f"\nunknown family falls back to the widest band: x/÷ {widest:.2f}")

rec = {
    "model": {"name": "sweep-mesh-gnn", "version": "0.3.0",
              "sha256": hashlib.sha256(open(CKPT,'rb').read()).hexdigest()},
    "coverage": 1 - ALPHA,
    "calibration": {"method": "mondrian_split_conformal_log_residual",
                    "set_size": len(res), "set_sha256": data.set_sha256,
                    "log_half_width": shared, "band_factor": math.exp(shared),
                    "delta": 0.0, "bands": bands},
    "normaliser": {"log_mean": G_MEAN, "log_std": G_STD, "delta": 0.0},
    "target": {"quantity": "von_mises_peak",
               "form": "dimensionless tau = sigma * L^2 / F",
               "L": "bounding-box diagonal",
               "to_physical": "sigma[MPa] = tau * F[N] / L[mm]^2"},
    "features": {"version": "sweep/v2", "columns": 13,
                 "layout": "pos(3) sdf(1) normal(3) is_fixed(1) is_loaded(1) "
                           "load_dir(3) poisson(1)",
                 "sdf": "signed distance to the part's own bounding box, "
                        "normalised by its diagonal"},
    "domain": {"material_class": "linear_elastic",
               "poisson_ratio": {"min": 0.27, "max": 0.42},
               "load_pattern": "single uniform traction on tagged faces",
               "families": sorted(bands), "source_dataset": "fea-surrogate sweep"},
    "limitations": [
        "constant-strain tetrahedra; peak reaches ~0.92 of a closed-form cantilever",
        "one load case per sample, not contact or moving loads",
        "linear elastic only",
        "a part whose family is not named gets the widest band, not the average",
        "screening estimate, not a strength certification",
    ],
    "trained_at": datetime.datetime.now(datetime.UTC).isoformat(),
}
json.dump(rec, open(f"{OUT}/calibration.json","w"), indent=2)
print("\nwrote", f"{OUT}/calibration.json")

## Reading the result

**The band.** Against SFEM's ×/÷ 19.75. Materially tighter means load location
was the binding constraint and the dataset is worth growing. Barely different
means it was not, and the next move is the model or the features rather than
twenty thousand more samples.

**The per-family table.** Each family's own band, and what the shared band
would have given it. The gap between those two columns is the cost of pretending
one band fits all.

**`calibration.json` now carries `bands`**, so the server selects per family at
inference. A part whose family is not named gets the widest band rather than the
average — an estimate whose group is unknown has not been shown to be covered at
the stated rate by anything narrower.
